In [1]:
from datasets import load_dataset

ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k")

/home/ys/diploma/denoising_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

class TestDataset(torch.utils.data.Dataset):
    def __init__(self,ds):
        super().__init__()
        self.ds = ds

    def __getitem__(self, index):
        
        return torch.tensor(self.ds[index]['noisy']['array']).to(torch.float32),  torch.tensor(self.ds[index]['clean']['array']).to(torch.float32), 
    
    def __len__(self):
        return self.ds.__len__()
    

test_set = TestDataset(ds['test'])
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, )
a = next(iter(test_loader))[0]
a.shape

torch.Size([1, 27861])

In [3]:
import cpuinfo
from time import perf_counter
from loaders import *
from lightning_modules.lightning_module import *
from utils.cfg_loader import load_cfg
import GPUtil
from torchmetrics.audio import( ScaleInvariantSignalDistortionRatio as SISDR, SignalDistortionRatio as SDR,
                                SignalNoiseRatio as SNR, ScaleInvariantSignalNoiseRatio as SISNR,
                                PerceptualEvaluationSpeechQuality as PESQ,
                                ShortTimeObjectiveIntelligibility as STOI)



cfg_path = 'configs/denoise_model_v1_cfg.yaml'
ckpt_path = '/home/ys/diploma/lightning_logs/version_30/checkpoints/best-checkpoint-epoch=00-valid_loss=0.79.ckpt'
#ckpt_path = '/home/ys/diploma/ckpts/last.ckpt'
model_cfg = load_cfg(cfg_path)

metrics = dict( pesq_wb = PESQ(16000, 'wb'),
                pesq_nb = PESQ(16000, 'nb'),
                stoi = STOI(16000),
                snratio = SNR(),
                sdratio = SDR(),
                sisdratio = SISDR(),
                sisnratio = SISNR())
                

model = UltraSpectrogramLightningModelUnet.load_from_checkpoint(ckpt_path, **model_cfg)

info = cpuinfo.get_cpu_info()
gpus = GPUtil.getGPUs()
print(" "*50)
print("-"*20 + 'DEVICE INFO' + "-"*20)
print(f"Процессор: {info['brand_raw']}")
print(f"Количество ядер: {info['count']}")
for g in gpus:
    print(f"Видеокарта: {g.name}")
    print(f"Память: {g.memoryTotal} MB")
    print(f"Используется памяти: {g.memoryUsed} MB")
    print(f"Загрузка GPU: {g.load * 100}%")
print("-"*20 + '----------' + "-"*20)   

def reset_metrics(metrics):
    """Сбрасывает все метрики перед новым вычислением."""
    for metric in metrics.values():
        metric.reset()

def compute_metrics(cleaned_wf, clean_wf, metric):
        cleaned_wf = cleaned_wf.detach().cpu()
        clean_wf = clean_wf.detach().cpu()
        cleaned_wf_shape = cleaned_wf.shape[-1]
        clean_wf_shape = clean_wf.shape[-1]
        if cleaned_wf.shape[1] != 1:
            cleaned_wf = cleaned_wf.sum(1, keepdims=True)
        if clean_wf.shape[1] != 1:
            clean_wf = clean_wf.sum(1, keepdims=True)
        if clean_wf_shape == min(clean_wf_shape, cleaned_wf_shape):
            cleaned_wf = cleaned_wf[:, :, :clean_wf_shape]
        else:
            clean_wf = clean_wf[:, :, :cleaned_wf_shape]

        
        for k in metric.keys():
            metric[k].update(cleaned_wf, clean_wf) 
        
        
def bench_model(model,metrics, loader, num_iters=100):
    model.eval()
    cpu = []
    gpu = []
    reset_metrics(metrics)
    with torch.no_grad():
        for i, batch in enumerate(loader):
            
            model = model.to('cpu')
            mixed, clean = batch
            
            mixed = mixed.to('cpu')
            
            if mixed.ndim == 2 and mixed.shape[0] > 1:
                mixed = mixed.sum(0, keepdim=True)
            
            if mixed.ndim == 3 and mixed.shape[1] > 1:
                mixed = mixed.sum(1, keepdim=True)

            if clean.ndim == 2 and clean.shape[0] > 1:
                clean = clean.sum(0, keepdim=True)
            
            if clean.ndim == 3 and clean.shape[1] > 1:
                clean = clean.sum(1, keepdim=True)

           
            
            start = perf_counter()
            out = model.run(mixed)
            delta =perf_counter() - start
            cpu.append(delta)
            
            model = model.to('cuda')
            mixed = mixed.to('cuda')
            clean = clean.to('cuda')
            
                
            start = perf_counter()

            out = model.run(mixed)
            delta = perf_counter() - start
            gpu.append(delta)

            out = out.to('cuda')

            if out.shape[-1] > clean.shape[-1]:
                out = out[..., :clean.shape[-1]]
        
            if clean.ndim != out.ndim:
                clean = clean[None, ...]

            compute_metrics(out, clean, metrics)
            
            if (i + 1) % (num_iters) == 0:
                break

        metric_values = {k: metrics[k].compute().item() for k in metrics.keys()}



    return cpu, gpu, metric_values


model


                                                  
--------------------DEVICE INFO--------------------
Процессор: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz
Количество ядер: 8
Видеокарта: NVIDIA GeForce GTX 1650
Память: 4096.0 MB
Используется памяти: 101.0 MB
Загрузка GPU: 0.0%
--------------------------------------------------


UltraSpectrogramLightningModelUnet(
  (stft): Spectrogram()
  (model): DenoisingModelUnet(
    (encoder): SpectrumEncoder(
      (encoder_features): Sequential(
        (layer_0): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(1, 8, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
            (depth_wise): Conv2d(8, 8, kernel_size=(9, 9), stride=(1, 1), groups=8)
            (point_wise): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), padding=(4, 4))
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ELU(alpha=1.0)
            (dropout): Dropout2d(p=0.2, inplace=False)
          )
          (scale): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (adapt_res): Conv2d(1, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (layer_1): AdaptiveResBlock(
          (block): MobileBlock(
            (conv): Conv2d(8, 32, kernel_size=(7, 7), stri

# VoiceBank + DEMAND

In [4]:
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader)


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snratio']}"),
print(f"SDR [dB]: {metric_values['sdratio']}")
print(f"SI-SDR [dB]: {metric_values['sisdratio']}")
print(f"SI-SNR [dB]: {metric_values['sisnratio']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesq_nb']}")
print(f"PESQ-WB: {metric_values['pesq_wb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.416770339012146
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.20639099180698395
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 3.9852163791656494
SDR [dB]: 5.169830799102783
SI-SDR [dB]: 5.031961441040039
SI-SNR [dB]: 5.032535552978516
STOI: 0.9042573571205139
PESQ-NB: 2.8375842571258545
PESQ-WB: 2.0457804203033447
---------------------------------------------
                                                  


In [5]:
audio = list(iter(test_loader))[17]

In [6]:
mixed = audio[0]
cleaned = model.run(mixed)

cleaned.shape

torch.Size([1, 1, 128000])

In [7]:
from IPython.display import Audio 

Audio(cleaned.detach().cpu().numpy()[0], rate=16000)



--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.048991084098816
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.26157259941101074
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 7.859231472015381
SDR [dB]: 10.862805366516113
SI-SDR [dB]: 6.800779342651367
SI-SNR [dB]: 6.801410675048828
STOI: 0.8085654377937317
PESQ-NB: 2.3817174434661865
PESQ-WB: 1.7474521398544312
---------------------------------------------

# LibreSpeech + Wham

In [5]:
_, _, test_loader = get_loaders(speech_dirs=["dataset/dev-clean", "dataset/test-clean"],
                                                    noise_dir="dataset/wham_noise/wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)

#print(next(iter(test_loader))[0].shape)
cpu, gpu,metric_values = bench_model(model, metrics, loader=test_loader)


print('-'*20 + '+CPU+' + '-'*20)
print('Avg CPU Inference [s] : ', torch.tensor(cpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*20 + '+GPU+' + '-'*20)
print('Avg GPU Inference [s] : ', torch.tensor(gpu).mean().item())
print('-'*20 + '-----' + '-'*20)
print('-'*19 + 'METRICS' + '-'*19 )
print(f"SNR [dB]: {metric_values['snratio']}"),
print(f"SDR [dB]: {metric_values['sdratio']}")
print(f"SI-SDR [dB]: {metric_values['sisdratio']}")
print(f"SI-SNR [dB]: {metric_values['sisnratio']}")
print(f"STOI: {metric_values['stoi']}")
print(f"PESQ-NB: {metric_values['pesq_nb']}")
print(f"PESQ-WB: {metric_values['pesq_wb']}")

print('-'*19 + '-------' + '-'*19 )
print(" "*50)

--------------------+CPU+--------------------
Avg CPU Inference [s] :  1.393979549407959
---------------------------------------------
--------------------+GPU+--------------------
Avg GPU Inference [s] :  0.2672978639602661
---------------------------------------------
-------------------METRICS-------------------
SNR [dB]: 7.743745803833008
SDR [dB]: 7.439727783203125
SI-SDR [dB]: 7.0822672843933105
SI-SNR [dB]: 7.086089611053467
STOI: 0.8968960642814636
PESQ-NB: 2.471888542175293
PESQ-WB: 1.8219600915908813
---------------------------------------------
                                                  


In [9]:
from IPython.display import Audio 
sample_batch = next(iter(test_loader))
mixed_waveforms, speech_waveforms = sample_batch

for i, (mixed_waveform, speech_waveform) in enumerate(zip(mixed_waveforms, speech_waveforms)):
    print(f"--------------------\nsample {i} from batch")
    print(f"Input audio {i + 1}:")
    display(Audio(mixed_waveform.numpy(), rate=16000))
    print(f"Target audio {i + 1}:")
    display(Audio(speech_waveform.numpy(), rate=16000))

--------------------
sample 0 from batch
Input audio 1:


Target audio 1:


In [10]:
torchaudio.save('examples/cleaned.wav', torch.tensor(out[0]), sample_rate=16000)
torchaudio.save('examples/clean.wav', torch.tensor(clean[0]), sample_rate=16000)
torchaudio.save('examples/noisy.wav', torch.tensor(mixed[0].detach().cpu()), sample_rate=16000)

NameError: name 'out' is not defined

In [ ]:
from torchvision.models import vgg11_bn, VGG11_BN_Weights
from torch import nn
import torch
_, valid_loader, test_loader = get_loaders(speech_dirs=["dataset/dev-clean", "dataset/test-clean"],
                                                    noise_dir="dataset/wham_noise/wham_noise",
                                                    batch_size=1,
                                                    padding_strategy=None)
aud1 = next(iter(valid_loader))[0].to('cuda')
aud2 = next(iter(test_loader))[0].to('cuda')
aud = torch.cat((aud1, aud2), dim=-1)
print(aud1.shape)
result = model.run(aud1[0])





torch.Size([1, 2, 128000])
torch.Size([1, 1, 128000])


In [10]:
from torchviz import make_dot
from torchview import draw_graph

model_graph = draw_graph(
    model.model,
    input_size=(1, 1, 1024, 1025),  
    device="cuda",
    expand_nested=True  # Показывает внутренности UNet
)
model_graph.visual_graph  # Отображает в ноутбуке
model_graph.visual_graph.render("unet_model", format="png") 

'unet_model.png'

In [ ]:
from IPython.display import Audio 
Audio(aud[0].numpy(),rate = 16000)

In [ ]:
Audio(result,rate = 16000) 

NameError: name 'Audio' is not defined

In [ ]:
!nvidia-smi

Sun Apr 13 12:49:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.72                 Driver Version: 566.14         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8              4W /   50W |     393MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
class PhaseCorrectorDilation(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=(3,5), padding='same'),
            nn.ELU(),
            nn.Conv2d(16, 16, kernel_size=(3,3), padding='same', dilation=(2,1)),
            nn.ELU(),
            nn.Conv2d(16, 1, kernel_size=3, padding='same'),
            nn.Tanh()
        )
        
    def forward(self, mag, noisy_phase):
        phase_diff = self.conv(torch.cat([mag, noisy_phase], dim=1))
        return noisy_phase + phase_diff
    

mag = torch.rand(4, 1, 1024, 1025)
noisy_phase = torch.rand(4, 1, 1024, 1025)
phs = PhaseCorrector()
phs(mag, noisy_phase).shape

torch.Size([4, 1, 1024, 1025])


torch.Size([4, 1, 1024, 1025])

In [ ]:
import torch.nn as nn
import torch
import torchaudio.functional as F
import torchaudio.transforms as T

class SiSDRLoss(nn.Module):
    def __init__(self, eps: float = 1e-9):
        """

        Args:
            eps (float): eps for stabibiluty calculations. Defaults to 1e-9.
        """
        super().__init__()
        self.eps = eps


    def forward(self, output, target):

        alpha = torch.sum(output * target, dim=-1,keepdim=True) / torch.norm(target, dim=-1)**2 

        proj = alpha * target

        proj_norm = torch.norm(proj, dim=-1)
        diff_norm = torch.norm((proj - output), dim=-1)

        return  -(10 * (torch.log10(proj_norm**2 / (diff_norm**2 + self.eps )))).mean()

class MultiResolutionLoss(nn.Module):
    def __init__(self, n_ftts: list = [256, 512, 1025, 2096]):
        super().__init__()
        self.nftts = n_ftts
    def forward(self, clean, enchanced):

        return torch.mean(torch.stack([nn.functional.l1_loss(T.Spectrogram(n_fft=n_fft, 
                                                                           power=1.0,
                                                                           normalized=True)(clean).log1p(),
                                                           T.Spectrogram(n_fft=n_fft, 
                                                                         power=1.0,
                                                                         normalized=True)(enchanced).log1p()) for n_fft in self.nftts]))

mrsl = MultiResolutionLoss()
sisdr = SiSDRLoss()

mrsl(aud1[:, 0 ,None,...], result[:, 0 ,None,...]), sisdr(aud1[:, 0 ,None,...], result[:, 0 ,None,...]) 

RuntimeError: stft input and window must be on the same device but got self on cuda:0 and window on cpu